# Context Switching

> **Task name:** `Context Switching`

**Track: Attention** | Does switching between task types within a sequence reduce accuracy?

In cognitive psychology, switching between task sets incurs a measurable cost in
accuracy and reaction time. This benchmark tests whether LLMs show analogous
switching costs when alternating between different task types within a single prompt.

Three conditions:
- **Pure blocks** — 60 items all of the same type (baseline)
- **Predictable alternation** — ABCABC... pattern with cues
- **Random switching** — random task order with explicit cues

**Metric:** Switch cost — accuracy drop from pure blocks to alternating/random conditions.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def parse_numbered_answers(response: str, count: int) -> list[str]:
    response = strip_thinking(response)
    lines = response.strip().split("\n")
    answers = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if re.match(r"^\d+[\.)\:\-]", line):
            cleaned = re.sub(r"^\d+[\.)\:\-]\s*", "", line)
            if cleaned:
                answers.append(cleaned)
    if not answers:
        for line in lines:
            line = line.strip()
            if line:
                answers.append(line)
    while len(answers) < count:
        answers.append("")
    return answers[:count]

def check_answer(expected: str, actual: str) -> bool:
    exp = expected.lower().strip()
    act = actual.lower().strip()
    if exp == act or exp in act:
        return True
    exp_d = re.sub(r"[,\s]", "", exp)
    act_d = re.sub(r"[,\s]", "", act)
    return exp_d == act_d and exp_d != ""

In [ ]:
# Three task types with seed data

# Task A: Number extraction
NUMBER_SENTENCES = [
    ("The shipment contained {n} crates of industrial equipment.", "{n}"),
    ("Researchers documented {n} distinct mineral deposits in the survey.", "{n}"),
    ("The bridge supports a maximum load of {n} tonnes per lane.", "{n}"),
    ("The greenhouse complex spans {n} hectares of farmland.", "{n}"),
    ("Inspectors cleared {n} aircraft for commercial service this quarter.", "{n}"),
    ("The reservoir holds approximately {n} megalitres of fresh water.", "{n}"),
    ("Emergency crews evacuated {n} residents from the flood zone.", "{n}"),
    ("The telescope captured images of {n} galaxies in a single exposure.", "{n}"),
    ("The expedition catalogued {n} butterfly species in the rainforest canopy.", "{n}"),
    ("Production lines assembled {n} vehicles during the overnight shift.", "{n}"),
    ("The census recorded {n} businesses operating within the district.", "{n}"),
    ("Divers recovered {n} artifacts from the wreck site.", "{n}"),
    ("The orchard yielded {n} bushels of pears this season.", "{n}"),
    ("Engineers installed {n} sensors throughout the monitoring network.", "{n}"),
    ("The festival attracted {n} performers from across the continent.", "{n}"),
]

# Task B: Country identification
CITY_SENTENCES = [
    ("The symposium in {city} drew scholars from leading institutions.", "{country}"),
    ("Construction crews in {city} completed the new transit terminal.", "{country}"),
    ("The heritage district of {city} was restored over a five-year project.", "{country}"),
    ("Agricultural exports from {city} increased by 18 percent.", "{country}"),
    ("The university in {city} opened a new marine biology research wing.", "{country}"),
    ("A renewable energy summit was held in {city} last November.", "{country}"),
    ("The national archives in {city} digitized over one million documents.", "{country}"),
    ("Public health officials in {city} launched a vaccination campaign.", "{country}"),
]

CITY_COUNTRY = [
    ("Tokyo", "Japan"), ("Paris", "France"), ("Cairo", "Egypt"),
    ("Buenos Aires", "Argentina"), ("Stockholm", "Sweden"),
    ("Lagos", "Nigeria"), ("Mumbai", "India"), ("Seoul", "South Korea"),
    ("Lima", "Peru"), ("Warsaw", "Poland"), ("Nairobi", "Kenya"),
    ("Bangkok", "Thailand"), ("Lisbon", "Portugal"), ("Hanoi", "Vietnam"),
    ("Prague", "Czech Republic"), ("Santiago", "Chile"), ("Accra", "Ghana"),
    ("Helsinki", "Finland"), ("Dublin", "Ireland"), ("Manila", "Philippines"),
    ("Oslo", "Norway"), ("Athens", "Greece"), ("Ankara", "Turkey"),
    ("Ottawa", "Canada"), ("Canberra", "Australia"), ("Bern", "Switzerland"),
    ("Dakar", "Senegal"), ("Riga", "Latvia"), ("Tallinn", "Estonia"),
    ("Zagreb", "Croatia"),
]

# Task C: Misspelling detection
MISSPELLING_SENTENCES = [
    ("The committee {word} the annual budget proposal.", "{misspelled}"),
    ("It was {word} to submit all forms before the deadline.", "{misspelled}"),
    ("The {word} impact of the regulation surprised analysts.", "{misspelled}"),
    ("Samples were processed in the {word} facility.", "{misspelled}"),
    ("The {word} hosted its annual awards ceremony.", "{misspelled}"),
    ("The study focused {word} on urban migration patterns.", "{misspelled}"),
    ("All {word} was inspected before deployment.", "{misspelled}"),
    ("Regular {word} prevented costly system failures.", "{misspelled}"),
]

MISSPELLINGS = [
    ("examined", "exmained"), ("necessary", "neccessary"), ("occurred", "occured"),
    ("separate", "seperate"), ("definitely", "definately"), ("environment", "enviroment"),
    ("temperature", "temprature"), ("laboratory", "labratory"), ("restaurant", "restaraunt"),
    ("government", "goverment"), ("beginning", "begining"), ("immediately", "immediatley"),
    ("professional", "proffesional"), ("independent", "independant"),
    ("apparently", "apparantly"), ("equipment", "equiptment"),
    ("maintenance", "maintainance"), ("technique", "tecnique"),
    ("sufficient", "sufficent"), ("accommodate", "accomodate"),
    ("committee", "commitee"), ("development", "developement"),
    ("knowledge", "knowlege"), ("guarantee", "guarentee"),
    ("permanent", "permenent"), ("privilege", "privelege"),
    ("rhythm", "rythm"), ("schedule", "scedule"),
    ("threshold", "threshhold"), ("vulnerable", "vunerable"),
]

TASK_CUES = {
    "number": "[NUMBER]",
    "country": "[COUNTRY]",
    "spelling": "[SPELLING]",
}

print(f"Seed data: {len(NUMBER_SENTENCES)} number templates, {len(CITY_SENTENCES)} city templates, {len(MISSPELLING_SENTENCES)} spelling templates")

In [ ]:
random.seed(654)
ITEMS_PER_SEQUENCE = 60
data = []
task_id = 0

def make_number_item(rng):
    template, ans_template = rng.choice(NUMBER_SENTENCES)
    n = rng.randint(10, 99999)
    n_str = f"{n:,}"
    sentence = template.format(n=n_str)
    answer = n_str
    return sentence, answer, "number"

def make_country_item(rng):
    template, _ = rng.choice(CITY_SENTENCES)
    city, country = rng.choice(CITY_COUNTRY)
    sentence = template.format(city=city)
    return sentence, country, "country"

def make_spelling_item(rng):
    template, _ = rng.choice(MISSPELLING_SENTENCES)
    correct, misspelled = rng.choice(MISSPELLINGS)
    sentence = template.format(word=misspelled)
    return sentence, misspelled, "spelling"

makers = {"number": make_number_item, "country": make_country_item, "spelling": make_spelling_item}

for variant_idx in range(2):  # 2 random seeds
    rng = random.Random(654 + variant_idx * 100)
    
    for condition in ["pure_number", "pure_country", "pure_spelling", "alternating", "random"]:
        items_text = []
        answers = []
        task_types = []
        is_switch = []  # True if this item switches from previous task type
        
        if condition.startswith("pure_"):
            task_type = condition.replace("pure_", "")
            for i in range(ITEMS_PER_SEQUENCE):
                sentence, answer, tt = makers[task_type](rng)
                cue = TASK_CUES[tt]
                items_text.append(f"{i+1}. {cue} {sentence}")
                answers.append(answer)
                task_types.append(tt)
                is_switch.append(False)
        
        elif condition == "alternating":
            cycle = ["number", "country", "spelling"]
            prev_tt = None
            for i in range(ITEMS_PER_SEQUENCE):
                tt = cycle[i % 3]
                sentence, answer, _ = makers[tt](rng)
                cue = TASK_CUES[tt]
                items_text.append(f"{i+1}. {cue} {sentence}")
                answers.append(answer)
                task_types.append(tt)
                is_switch.append(prev_tt is not None and tt != prev_tt)
                prev_tt = tt
        
        elif condition == "random":
            prev_tt = None
            for i in range(ITEMS_PER_SEQUENCE):
                tt = rng.choice(["number", "country", "spelling"])
                sentence, answer, _ = makers[tt](rng)
                cue = TASK_CUES[tt]
                items_text.append(f"{i+1}. {cue} {sentence}")
                answers.append(answer)
                task_types.append(tt)
                is_switch.append(prev_tt is not None and tt != prev_tt)
                prev_tt = tt
        
        prompt = (
            f"Below are {ITEMS_PER_SEQUENCE} items. Each has a task cue in brackets:\n"
            "- [NUMBER]: Extract the number mentioned\n"
            "- [COUNTRY]: Name the country of the city referenced\n"
            "- [SPELLING]: Identify the misspelled word\n\n"
            "Follow the cue for EACH item. Respond with one answer per line, numbered to match.\n\n"
            + "\n".join(items_text)
            + "\n\nAnswers:"
        )
        
        data.append({
            "task_id": task_id,
            "prompt": prompt,
            "answers": answers,
            "task_types": task_types,
            "is_switch": is_switch,
            "condition": condition,
            "variant": variant_idx,
            "num_items": ITEMS_PER_SEQUENCE,
        })
        task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} sequences")
print(f"By condition: {dict(df_all['condition'].value_counts())}")

## Task Definition

In [ ]:
@kbench.task(
    name="context_switching",
    description="Measure task-switching cost when alternating between number, country, and spelling tasks"
)
def context_switching(
    llm,
    prompt: str,
    answers: str,
    num_items: int,
) -> tuple[int, int]:
    response = llm.prompt(prompt)
    expected = json.loads(answers) if isinstance(answers, str) else answers
    n = int(num_items)
    parsed = parse_numbered_answers(response, n)
    correct = sum(1 for e, a in zip(expected, parsed) if check_answer(e, a))
    return (correct, n)

## Run Evaluation

Uses `kbench.llm` — the default model. Use Kaggle's **"Add Models"** button to run across multiple models.

In [ ]:
eval_df = df_all.copy()
eval_df["answers"] = eval_df["answers"].apply(json.dumps)
task_eval_df = eval_df[["prompt", "answers", "num_items"]].copy()

print(f"Running {len(task_eval_df)} context-switching sequences with kbench.llm...")
runs = context_switching.evaluate(
    llm=kbench.llm,
    evaluation_data=task_eval_df,
    n_jobs=2,
    timeout=180,
    max_attempts=2,
)
results = runs.as_dataframe()
results["accuracy"] = results["result"].apply(
    lambda r: r[0] / r[1] if isinstance(r, tuple) and r[1] > 0 else float(r) if not isinstance(r, tuple) else 0
)
results = results.merge(
    eval_df[["prompt", "condition", "variant", "task_types", "is_switch"]],
    on="prompt", how="left"
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
print("=== Accuracy by Condition ===")
cond_stats = results.groupby("condition")["accuracy"].agg(["mean", "std", "count"])
print(cond_stats.to_string(float_format="%.3f"))

# Compute switch costs
pure_acc = results[results["condition"].str.startswith("pure_")]["accuracy"].mean()
alt_acc = results[results["condition"] == "alternating"]["accuracy"].mean()
rand_acc = results[results["condition"] == "random"]["accuracy"].mean()

print(f"\n=== Switch Costs ===")
print(f"Pure blocks baseline: {pure_acc:.2%}")
print(f"Alternating:          {alt_acc:.2%} (cost: {pure_acc - alt_acc:+.2%})")
print(f"Random switching:     {rand_acc:.2%} (cost: {pure_acc - rand_acc:+.2%})")

# Per-task-type accuracy in mixed conditions
print("\n=== Per-Task-Type Accuracy in Mixed Conditions ===")
for _, row in results[results["condition"].isin(["alternating", "random"])].iterrows():
    task_types = json.loads(row["task_types"]) if isinstance(row["task_types"], str) else row["task_types"]
    answers_exp = json.loads(row.get("answers", "[]")) if isinstance(row.get("answers"), str) else []
    parsed = parse_numbered_answers(str(row.get("response", "")), len(task_types))
    
    for tt in ["number", "country", "spelling"]:
        indices = [i for i, t in enumerate(task_types) if t == tt]
        if indices and answers_exp:
            tt_correct = sum(1 for i in indices if i < len(answers_exp) and i < len(parsed) 
                           and check_answer(answers_exp[i], parsed[i]))
            tt_total = len(indices)
            print(f"  {row['condition']} / {tt}: {tt_correct}/{tt_total} = {tt_correct/tt_total:.2%}")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

conditions = ["pure_number", "pure_country", "pure_spelling", "alternating", "random"]
labels = ["Pure\nNumber", "Pure\nCountry", "Pure\nSpelling", "Alternating\nABCABC", "Random\nSwitch"]
colors = ["steelblue", "steelblue", "steelblue", "darkorange", "tomato"]

means = [results[results["condition"] == c]["accuracy"].mean() for c in conditions]
stds = [results[results["condition"] == c]["accuracy"].std() for c in conditions]

bars = ax.bar(labels, means, yerr=stds, color=colors, capsize=5, edgecolor="black", linewidth=0.5)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f"{mean:.0%}", ha="center", fontsize=11, fontweight="bold")

# Add switch cost annotation
if pure_acc > 0:
    ax.axhline(y=pure_acc, color="gray", linestyle="--", alpha=0.5, label=f"Pure baseline ({pure_acc:.0%})")

ax.set_ylabel("Accuracy")
ax.set_title("Context Switching: Task-Switch Cost")
ax.set_ylim(0, 1.15)
ax.legend()
plt.tight_layout()
plt.savefig("context_switching_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to context_switching_results.png")